In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
from PIL import Image
import h5py

from scipy.stats import zscore

In [ ]:
fmri_data_dir = "${MBS_THINGS_RAW_DIR}/fmri"

In [ ]:
SUBJECTS = [f"sub-{s:02d}" for s in range(1, 4)]

In [ ]:
ROIs ={
    'whole_brain': ['whole_brain'],
    'V1': ['V1'],
    'V2': ['V2'],
    'V3': ['V3'],
    'V4': ['hV4'],
    'VO1': ['VO1'],
    'VO2': ['VO2'],
    'LO1': ['LO1 (prf)'],
    'LO2': ['LO2 (prf)'],
    'TO1': ['TO1'],
    'TO2': ['TO2'],
    'V3b': ['V3b'],
    'V3a': ['V3a'],
    # 'EBA': ['lEBA', 'rEBA'],
    # 'FFA': ['lFFA', 'rFFA'],
    # 'OFA': ['lOFA', 'rOFA'],
    # 'STS': ['lSTS', 'rSTS'],
    # 'PPA': ['lPPA', 'rPPA'],
    # 'RSC': ['lRSC', 'rRSC'],
    # 'TOS': ['lTOS', 'rTOS'],
    # 'LOC': ['lLOC', 'rLOC'],
    'lEBA': ['lEBA'],
    'rEBA': ['rEBA'],
    'lFFA': ['lFFA'],
    'rFFA': ['rFFA'],
    'lOFA': ['lOFA'],
    'rOFA': ['rOFA'],
    # 'lSTS': ['lSTS'],
    # 'rSTS': ['rSTS'],
    'lPPA': ['lPPA'],
    'rPPA': ['rPPA'],
    'lRSC': ['lRSC'],
    'rRSC': ['rRSC'],
    'lTOS': ['lTOS'],
    'rTOS': ['rTOS'],
    'lLOC': ['lLOC'],
    'rLOC': ['rLOC'],
    'IT': ['IT'],
    'IT_glasser': [
    'glasser-TE1p','glasser-TE2p', 'glasser-FFC', 'glasser-VVC', 'glasser-VMV2', 
    'glasser-VMV3', 'glasser-PHA1', 'glasser-PHA2', 'glasser-PHA3'
    ]
}

In [ ]:
# NOISE_CEILING_THRESHOLD = 0.3
NOISE_CEILING_THRESHOLD = 0.1

### Load neural data

In [ ]:
def load_subject_data(fmri_data_dir: Union[str, Path], sub: str, drop_voxel_id_from_responses=True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load the fMRI data and associated metadata for a given subject.
    """
    if not isinstance(fmri_data_dir, Path):
        fmri_data_dir = Path(fmri_data_dir)
        
    if not str(fmri_data_dir).endswith('/betas_csv'):
        fmri_data_dir = fmri_data_dir / "betas_csv"
    
    # Load the metadata
    metadata_file = fmri_data_dir / f"{sub}_StimulusMetadata.csv"
    metadata = pd.read_csv(metadata_file)

    # Load the fMRI data
    responses_file = fmri_data_dir / f"{sub}_ResponseData.h5"
    responses = pd.read_hdf(responses_file)
    if drop_voxel_id_from_responses:
        responses = responses.drop(columns=['voxel_id'])
    
    # Load the voxel data
    voxdata_file = fmri_data_dir / f"{sub}_VoxelMetadata.csv"
    voxdata = pd.read_csv(voxdata_file)
    

    return responses, metadata, voxdata




In [ ]:
selected_columns = [
    'voxel_x', 'voxel_y', 'voxel_z',
    'nc_singletrial', 'nc_testset', 
    'splithalf_uncorrected', 'splithalf_corrected'
]

def process_subject_data(responses: pd.DataFrame, metadata: pd.DataFrame, voxdata: pd.DataFrame, ROIs:Dict[str, List[str]], noise_ceiling_threshold:float=None) \
    -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray, np.ndarray, np.ndarray]:
    """
    Process the fMRI data for a given subject.
    """

    # Get sorting indices w.r.t. stimulus filenames
    # and split the data into train and test sets
    indices = metadata.sort_values(['stimulus', 'trial_id']).index
    train_mask = ((metadata['trial_type'] == 'train').values)[indices]
    train_indices, test_indices = indices[train_mask], indices[~train_mask]
    
    # Get the train and test stimuli
    train_stimuli = metadata['stimulus'][train_indices].values
    test_stimuli = metadata['stimulus'][test_indices].values
    test_stimuli = test_stimuli.reshape(-1, 12)
    
    assert np.all(test_stimuli == test_stimuli[:, [0]])
    test_stimuli = test_stimuli[:, 0]
    
    # Get the train and test responses
    train_responses, test_responses = {}, {}
    noise_ceilings = {}
    for roi_name, roi_list in ROIs.items():
        # Get the voxel data for the current ROI
        if roi_name == 'whole_brain':
            voxel_mask = np.ones(len(voxdata), dtype=bool)
        else:
            voxel_mask = voxdata[roi_list].sum(axis=1).values.astype(bool)

        noise_ceiling = voxdata.loc[:, 'nc_testset'].to_numpy()
        # If a noise ceiling threshold is provided, filter out the voxels
        # that do not meet the threshold
        if noise_ceiling_threshold:
            noise_ceiling_mask = noise_ceiling > noise_ceiling_threshold * 100
            voxel_mask = voxel_mask & noise_ceiling_mask
            noise_ceilings[roi_name] = noise_ceiling[voxel_mask]
        
        # Get the fMRI responses for the selected voxels
        roidata = responses[voxel_mask].to_numpy().T
        
        # Get the train and test responses for the current ROI
        train_responses_roi = roidata[train_indices]
        test_responses_roi = roidata[test_indices]
            
        train_responses[roi_name] = train_responses_roi
        test_responses[roi_name] = test_responses_roi
        
        voxel_metadata = voxdata[selected_columns]
    
    return train_responses, test_responses, train_stimuli, test_stimuli, voxel_mask, voxel_metadata, noise_ceilings
    
    

In [ ]:
responses_subjects, metadata_subjects, voxdata_subjects = {}, {}, {}
for sub in tqdm(SUBJECTS):
    responses, metadata, voxdata = load_subject_data(fmri_data_dir, sub)
    responses_subjects[sub] = responses
    metadata_subjects[sub] = metadata
    voxdata_subjects[sub] = voxdata

In [ ]:
metadata_subjects[SUBJECTS[0]].keys()
# responses_subjects[SUBJECTS[0]]
x = voxdata_subjects[SUBJECTS[0]]
x.columns

In [ ]:
c = 0.1
for sub in SUBJECTS:
    voxdata = voxdata_subjects[sub]
    print(sub, (voxdata['nc_testset'] >= c * 100).sum())

In [ ]:
# raise

In [ ]:
# responses_subjects[subjects[0]]
# metadata_subjects[subjects[0]].keys()

In [ ]:

train_responses_subjects, test_responses_subjects = {}, {}
train_responses_concatenated, test_responses_concatenated = {}, {}
noise_ceilings_subjects = {}
train_stimuli, test_stimuli = None, None

for sub in tqdm(SUBJECTS):
    responses, metadata, voxdata = responses_subjects[sub], metadata_subjects[sub], voxdata_subjects[sub]
    # Check: stimuli filename is of the form `{concept}_id.jpg`
    stimuli, concepts = metadata['stimulus'].values, metadata['concept'].values
    assert all([stimulus.rsplit('_', 1)[0] == concept for stimulus, concept in zip(stimuli, concepts)])
    
    # Process the data for the current subject
    train_responses_, test_responses_, train_stimuli_, test_stimuli_, voxel_mask_, voxel_metadata_, noise_ceilings_ = \
        process_subject_data(responses, metadata, voxdata, ROIs, noise_ceiling_threshold=NOISE_CEILING_THRESHOLD)
        
    noise_ceilings_subjects[sub] = noise_ceilings_
        
    for region,region_responses in train_responses_.items():
        train_responses_[region] = np.expand_dims(region_responses, -1).mean(axis=-1)
        
    for roi_name, roi_responses in test_responses_.items():
        roi_responses = roi_responses.transpose()
        new_shape = (roi_responses.shape[0], roi_responses.shape[1] // 12, 12)
        roi_responses = roi_responses.reshape(new_shape)
        test_responses_[roi_name] = roi_responses.transpose((1, 0, 2)).mean(axis=-1)
    
    train_responses_subjects[sub] = train_responses_
    test_responses_subjects[sub] = test_responses_
    
    
    if train_stimuli is None:
        train_stimuli = train_stimuli_
        test_stimuli = test_stimuli_
    else:
        # Check: stimuli are in the same order across subjects
        assert np.array_equal(train_stimuli, train_stimuli_)
        assert np.array_equal(test_stimuli, test_stimuli_)



In [ ]:
roi = list(ROIs)[0]
train_responses_subjects['sub-01'][roi].shape

In [ ]:

# Average the test responses across the 12 repetitions
# test_responses_concatenated_avg = {}
# for roi_name, roi_responses in test_responses_concatenated.items():
#     new_shape = (roi_responses.shape[0] // 12, roi_responses.shape[1], 12)
#     test_responses_concatenated_avg[roi_name] = np.mean(roi_responses.reshape(new_shape), axis=-1)

In [ ]:
train_responses_subjects_norm = {}
test_responses_subjects_norm = {}

# Standardize the responses to have mean 0 and std 1
for subj in tqdm(SUBJECTS):
    print(f"Normalizing subject {subj}...")
    train_responses_subjects_norm[subj] = {}
    test_responses_subjects_norm[subj] = {}
    for roi_name, roi_responses in tqdm(train_responses_subjects[subj].items(), leave=False, total=len(ROIs)):
        # mean = roi_responses.mean(axis=(0), keepdims=True)
        # std = roi_responses.std(axis=(0), keepdims=True)
        # train_responses_subjects_norm[subj][roi_name] = (roi_responses - mean) / std
        # test_responses_subjects_norm[subj][roi_name] = ((test_responses_subjects[subj][roi_name] - mean) / std)
        
        # train_responses_subjects_norm[subj][roi_name] = (roi_responses - mean) / std
        # test_responses_subjects_norm[subj][roi_name] = ((test_responses_subjects[subj][roi_name] - mean[..., None]) / std[..., None]).mean(axis=-1)
        
        # train_responses_subjects_norm[subj][roi_name] = roi_responses
        # test_responses_subjects_norm[subj][roi_name] = test_responses_subjects[subj][roi_name]
        
        train_responses_subjects_norm[subj][roi_name] = zscore(roi_responses, axis=0)
        test_responses_subjects_norm[subj][roi_name] = zscore(test_responses_subjects[subj][roi_name], axis=0)
        
        print(f"\t {roi_name}: train mean {train_responses_subjects_norm[subj][roi_name].mean():.4f}, "
                f"std {train_responses_subjects_norm[subj][roi_name].std(axis=0).mean():.4f}; "
                f"test mean {test_responses_subjects_norm[subj][roi_name].mean():.4f}, "
                f"std {test_responses_subjects_norm[subj][roi_name].std(axis=0).mean():.4f}")

In [ ]:
# resp = train_responses_concatenated['IT']
# sns.kdeplot(data=resp.std(axis=0), bw_adjust=0.5)
# plt.xticks(np.arange(0.9, 1.1, 0.1))

In [ ]:
for subj in SUBJECTS:    
    for roi_name in ROIs.keys():
        print(f"Subject: {subj} - ROI: {roi_name}")
        print(f"Train responses shape: {train_responses_subjects_norm[subj][roi_name].shape}")
        print(f"Test responses shape: {test_responses_subjects_norm[subj][roi_name].shape}")
        print(f"Train stimuli shape: {train_stimuli.shape}")
        print(f"Test stimuli shape: {test_stimuli.shape}")
        print(f"Noise ceilings shape: {noise_ceilings_subjects[subj][roi_name].shape}")
        print()

In [ ]:
train_stimulus_ids = [f"{stimulus.rsplit('_', 1)[0]}/{stimulus}" for stimulus in train_stimuli]
test_stimulus_ids = [f"{stimulus.rsplit('_', 1)[0]}/{stimulus}" for stimulus in test_stimuli]

len(train_stimulus_ids), len(test_stimulus_ids), train_stimulus_ids[0], test_stimulus_ids[0]

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": train_stimulus_ids,
            "neural_data": train_responses_subjects_norm,
        },
    "test" :
        {
            "stimulus_ids": test_stimulus_ids,
            "neural_data": test_responses_subjects_norm,
        },
    "noise_ceilings": noise_ceilings_subjects
}

### Metadata

In [ ]:
metadata = {
    "noise_ceiling_threshold": NOISE_CEILING_THRESHOLD,
    "ROIs": ROIs,
    "desc": f"""
    The neural data is from the THINGS dataset, recorded from 3 humans.
    The neural data is recorded from V1, V2, V4, and IT ROIs.
    The dataset includes both training and Test splits.
    Voxels with noise ceiling less than {NOISE_CEILING_THRESHOLD} are excluded.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = f'things_fmri_train_test_bench-whole_brain-nc_{int(NOISE_CEILING_THRESHOLD * 100):02d}.h5'
filename = f'things_fmri.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

if not data_dir.exists():
    data_dir.mkdir(parents=False, exist_ok=False)

In [ ]:
filename

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi in tqdm(ROIs.keys(), leave=False, total=len(ROIs), desc=f"{split} - {subj}"):
                f.create_dataset(f"{split}/neural_data/{subj}/{roi}", data=processed_data[split]['neural_data'][subj][roi])
                
                
    for subj in SUBJECTS:
        for roi in ROIs.keys():
            f.create_dataset(f"noise_ceilings/{subj}/{roi}", data=processed_data['noise_ceilings'][subj][roi])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = list(ROIs.keys())
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    for split in ['train', 'test']:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in SUBJECTS:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in ROIs.keys():
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in SUBJECTS:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in ROIs.keys():
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['sub-01'][list(ROIs)[0]].shape
loaded_data['noise_ceilings']['sub-01'][list(ROIs)[0]].shape

In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    for split in ['train', 'test']:
        
        loaded_data[split]['neural_data'] = {}
        for subj in SUBJECTS:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in ROIs.keys():
                x = f[split]['neural_data'][subj][roi][()]
                mean, std = x.mean(axis=(0,)), x.std(axis=(0,))
                print(f"{split} - {subj} - {roi} - mean: {mean.mean()}, std: {std.mean()}")


In [ ]:

# Check the saved data
with h5py.File(data_path, 'r') as f:
    print(f.keys())
    for split in ['train', 'test']:
        print(split)
        print(f[split]['stimulus_ids'].shape)

        for subj in SUBJECTS:
            for roi in ROIs.keys():
                print(f[split]['neural_data'][subj][roi].shape)
                
    for subj in SUBJECTS:
        for roi in ROIs.keys():
            print(f['noise_ceilings'][subj][roi].shape)
                
    print(json.loads(f.attrs['metadata']))

In [ ]:
# roi = 'IT'
# split = 'train'
# with h5py.File(data_path, 'r') as f:
#     resp = f[split]['neural'][roi][()]
    
# plt.hist(resp.std(axis=0), bins=50)

# Control